In [ ]:
import pandas as pd
from utils import reduce_memory_usage
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1-0. 데이터 로드
data_path = '../data/prep/total_data.csv' 
data = pd.read_csv(data_path)
data = reduce_memory_usage(data)

print(f"전체 데이터 형태: {data.shape}")

# 1-1. 학습 데이터 추출
train_df = data[data['eval_set'] == 'train']

# 1-2. 전체 피처와 전체 정답 분리
unused_cols = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered',
               'order_number'] # 모델 학습에 사용하지 않을 컬럼들
all_feat = train_df.drop(columns=unused_cols).select_dtypes(include=['number']).fillna(0) # 숫자형 데이터만 추출, 결측치는 0으로 채움
all_labels = train_df['reordered'] # 실제로 샀는지 안 샀는지, 재구매 여부 (정답)

# 1-3. 스케일링 (정규화)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(all_feat) # 수치형 피처들만 스케일링

# 1-4. 학습용(Train) / 검증용(Val) 데이터 분리 (8:2)
# [훈련피처, 검증피처, 훈련정답, 검증정답]
train_feat, val_feat, train_labels, val_labels = train_test_split(
    train_scaled, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

print(f"학습 피처: {train_feat.shape}, 학습 정답: {train_labels.shape}")
print(f"검증 피처: {val_feat.shape}, 검증 정답: {val_labels.shape}")

In [ ]:
import numpy as np

# 1. 정답 데이터(y)의 클래스 비중 확인
print("[검증 데이터(Val) 내 재구매(1) vs 미구매(0) 비율]")
unique, counts = np.unique(val_labels, return_counts=True)
ratio = dict(zip(unique, counts))
for cls, count in ratio.items():
    print(f"클래스 {cls} (0:미구매, 1:재구매): {count}건 ({count/len(val_labels)*100:.2f}%)")

# 2. 학습 데이터(X)와 정답(y)의 짝이 맞는지 상위 5개 샘플로 대조
print("\n[데이터-정답 매칭 확인 (상위 5개)]")
for i in range(5):
    # 피처 중 재구매와 상관관계가 높은 'prod_reorder_cnt' 스케일링 전 값을 확인
    print(f"Sample {i+1}: 피처 요약(평균값) {train_feat[i].mean():.4f}  ===>  정답(reordered): {train_labels.iloc[i]}")

# 3. 셔플링 확인 (데이터가 편향되게 몰려있지 않고 잘 섞였는지)
print("\n[정답 데이터의 앞부분 20개 샘플링]")
print(train_labels.values[:20])

In [ ]:
from sklearn.linear_model import LogisticRegression

# 2-1. 모델 객체 생성
# solver='saga': 확률적 경사하강법 상위 버전, 대용량 데이터 처리에 최적화된 알고리즘
# max_iter: 최대 업데이트 정도
model = LogisticRegression(solver='saga', max_iter=200, random_state=42)

# 2-2. 모델 학습 시작
model.fit(train_feat, train_labels)

In [ ]:
# 학습 세트 정확도
print(model.score(train_feat, train_labels))

# 검증 세트 정확도
print(model.score(val_feat, val_labels))

# 각 feature들의 계수 확인 (어떤 피처가 재구매 예측에 큰 영향을 주는지)
print(model.coef_)

In [ ]:
import pandas as pd

# 1. 계수(Coefficient)와 컬럼명을 매칭하여 데이터프레임 생성
# model.coef_[0]은 로지스틱 회귀의 가중치 값들입니다.
coef_df = pd.DataFrame({
    'Feature': all_feat.columns,
    'Coefficient': model.coef_[0]
})

# 2. 계수값이 소수점까지 완전히 동일한 그룹 찾기
# 동일한 계수를 가진 행들을 리스트로 묶습니다.
duplicates = coef_df.groupby('Coefficient')['Feature'].apply(list).reset_index()

# 3. 2개 이상의 변수가 같은 계수를 가진 경우만 필터링
duplicate_groups = duplicates[duplicates['Feature'].map(len) > 1].sort_values(by='Coefficient', ascending=False)

print("[계수값이 중복된 변수 그룹 확인]")
if not duplicate_groups.empty:
    for idx, row in duplicate_groups.iterrows():
        print(f"\n계수값: {row['Coefficient']}")
        print(f"해당 변수들: {row['Feature']}")
else:
    print("중복된 계수값을 가진 변수가 없습니다.")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# 성능 평가
# 1. 확률값 및 예측값 계산
val_probs = model.predict_proba(val_feat)[:, 1]
custom_threshold = 0.2
val_preds_custom = (val_probs >= custom_threshold).astype(int)

# 2. 혼동 행렬을 이용한 상세 지표 계산
tn, fp, fn, tp = confusion_matrix(val_labels, val_preds_custom).ravel()

sensitivity = tp / (tp + fn)       # Recall
specificity = tn / (tn + fp)
pos_pred_value = tp / (tp + fp)    # Precision
neg_pred_value = tn / (tn + fn)
prevalence = (tp + fn) / (tp + tn + fp + fn)
detection_rate = tp / (tp + tn + fp + fn)
detection_prevalence = (tp + fp) / (tp + tn + fp + fn)
balanced_accuracy = (sensitivity + specificity) / 2

# 3. 결과 출력 (구분선 제거 및 형식 맞춤)
print(f"[임계값 {custom_threshold} 적용 결과]")
print(f"Accuracy: {accuracy_score(val_labels, val_preds_custom):.4f}")
print(f"F1-Score: {f1_score(val_labels, val_preds_custom):.4f}")

print("\n[상세 분류 리포트]")
print(f"            Sensitivity : {sensitivity:.5f}") # 민감도
print(f"            Specificity : {specificity:.5f}") # 특이도
print(f"         Pos Pred Value : {pos_pred_value:.5f}") # 정밀도
print(f"         Neg Pred Value : {neg_pred_value:.5f}") # 재현율
print(f"             Prevalence : {prevalence:.5f}") # 유세율
print(f"         Detection Rate : {detection_rate:.5f}") # 탐지율
print(f"   Detection Prevalence : {detection_prevalence:.5f}") # 탐지 유세율
print(f"      Balanced Accuracy : {balanced_accuracy:.5f}") # 균형 정확도

print("\n[기본 분류 리포트]")
print(classification_report(val_labels, val_preds_custom, target_names=['미구매(0)', '재구매(1)']))

print("\n[임계값별 F1-Score 변화]")
for t in [0.1, 0.15, 0.2, 0.25, 0.3]:
    preds = (val_probs >= t).astype(int)
    print(f"Threshold {t:.2f} -> F1-Score: {f1_score(val_labels, preds):.4f}")